# Smart Home Energy Consumption Prediction and Optimization
**University Machine Learning Innovative Assignment**  
**Domain:** Smart Energy Management  
**Dataset:** UCI Appliances Energy Prediction (`energydata_complete.csv`)  
**Task:** Predict household appliance energy consumption (Wh) for the next 10-minute interval ($t+1$) using information available at or before current time $t$.

## 1. Imports and Setup

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src directory to path
sys.path.append(os.path.abspath('../src'))
from data_preparation import load_and_prepare_data
from preprocessing import analyze_target_distribution, chronological_split
from feature_engineering import create_features
from predict import EnergyPredictor

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Imports successful!')

## 2. Stage 2 & 3: Data Loading & Preparation
We inspect raw data structure, verify strict 10-minute sampling intervals, sort chronologically, and ensure zero duplicate timestamps.

In [ ]:
raw_path = '../data/raw/energydata_complete.csv'
processed_path = '../data/processed/clean_energydata.csv'
df_clean = load_and_prepare_data(raw_path, processed_path)
df_clean.head()

## 3. Stage 4 & 5: Data Preprocessing & Target Distribution Analysis
Here we verify that classification balancing techniques (such as SMOTE) are mathematically inapplicable to continuous Wh regression, and demonstrate why natural energy spikes must be retained.

In [ ]:
target_analysis = analyze_target_distribution(df_clean)
print(json.dumps(target_analysis, indent=2))

## 4. Stage 6: Exploratory Data Analysis (Key Visual Insights)

In [ ]:
# Display generated publication figures from outputs/figures
from IPython.display import Image, display

figures_to_show = [
    '../outputs/figures/fig1_appliance_distribution.png',
    '../outputs/figures/fig3_hourly_consumption_pattern.png',
    '../outputs/figures/fig7_correlation_heatmap.png',
    '../outputs/figures/fig8_high_consumption_zoom.png'
]

for fig_path in figures_to_show:
    if os.path.exists(fig_path):
        display(Image(filename=fig_path))
    else:
        print(f'Figure {fig_path} not found. Run src/eda.py to generate.')

## 5. Stage 7: Feature Engineering & Leakage Prevention
We construct cyclical time encodings, past consumption lags (10m, 20m, 30m, 1h, 24h), historical rolling statistics, and domain thermal gradients. Random noise variables (`rv1`, `rv2`) are intentionally removed.

In [ ]:
featured_path = '../data/processed/featured_energydata.csv'
df_feat = pd.read_csv(featured_path)
print(f'Engineered features shape: {df_feat.shape}')
print('Engineered features list:', df_feat.columns.tolist()[:15], '...')

## 6. Stage 8, 9, 10, 11: Model Training, Tuning & Evaluation
Compare Baselines (Mean, Persistence) against Machine Learning Models (Ridge Regression, Random Forest, HistGradientBoosting).

In [ ]:
metrics_path = '../outputs/metrics/model_comparison.csv'
if os.path.exists(metrics_path):
    metrics_df = pd.read_csv(metrics_path)
    print('--- Model Comparison Table ---')
    display(metrics_df)
else:
    print('Run src/train.py first to generate metrics.')

## 7. Stage 11: Diagnostic Plots & Error Dissection

In [ ]:
eval_figs = [
    '../outputs/figures/eval_fig1_actual_vs_predicted.png',
    '../outputs/figures/eval_fig2_residual_distribution.png',
    '../outputs/figures/eval_fig4_model_comparison_bars.png',
    '../outputs/figures/eval_fig6_high_consumption_error_analysis.png'
]

for fig_path in eval_figs:
    if os.path.exists(fig_path):
        display(Image(filename=fig_path))
    else:
        print(f'Figure {fig_path} not found. Run src/evaluation.py to generate.')

## 8. Real-Time Inference & Energy Cost Optimization Demonstration

In [ ]:
predictor = EnergyPredictor(model_path='../models/best_model.joblib', features_path='../models/feature_names.json')
test_preds = pd.read_csv('../outputs/metrics/test_predictions.csv')

# Select real test sample
sample = test_preds.iloc[100].to_dict()
prediction_result = predictor.predict(sample, tariff_per_kwh=0.18)

print('Actual Next Consumption:   ', sample['target_next_appliances'], 'Wh')
print('Predicted Next Consumption:', prediction_result['predicted_consumption_Wh'], 'Wh')
print('Predicted Next (kWh):      ', prediction_result['predicted_consumption_kWh'], 'kWh')
print('Estimated Interval Cost:   ', f"${prediction_result['estimated_interval_cost']:.4f}")
print('95% Confidence Interval:   ', prediction_result['prediction_interval_95_Wh'], 'Wh')
print('Advisory Recommendation:   ', prediction_result['exploratory_suggestion'])